# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import chromadb

from dotenv import load_dotenv
from tavily import TavilyClient
from pydantic import BaseModel

from lib.agents import Agent
from lib.llm import LLM
from lib.tooling import tool

# Load environment variables
from pathlib import Path
load_dotenv(Path.cwd().parent.parent / "config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Connect to the existing ChromaDB from Part 1
embedding_fn = chromadb.utils.embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-3-large"
)

chroma_client = chromadb.PersistentClient(path="chroma_db")

collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [3]:
class EvaluationReport(BaseModel):
    useful: bool
    description: str

In [4]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str):
    """
    Semantic search: Finds the most relevant game information in the vector database.

    Args:
        query: A question about the game industry.

    Returns:
        A list of the most relevant games including platform, publisher,
        release year, genre and description.
    """

    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    retrieved_games = []

    for metadata in results["metadatas"][0]:
        retrieved_games.append(metadata)

    return retrieved_games

In [6]:
retrieve_game("Who published Minecraft?")

[{'Name': 'Minecraft',
  'Description': 'A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adventure.',
  'Publisher': 'Mojang Studios',
  'Genre': 'Sandbox, Survival',
  'Platform': 'Xbox One',
  'YearOfRelease': 2014},
 {'YearOfRelease': 2021,
  'Description': "The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting.",
  'Name': 'Halo Infinite',
  'Publisher': 'Xbox Game Studios',
  'Platform': 'Xbox Series X|S',
  'Genre': 'First-person shooter'},
 {'Genre': 'Platformer',
  'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
  'Publisher': 'Nintendo',
  'Platform': 'Nintendo 64',
  'YearOfRelease': 1996,
  'Name': 'Super Mario 64'}]

#### Evaluate Retrieval Tool

In [7]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: list):
    """
    Based on the user's question and the retrieved documents,
    evaluate whether the retrieved documents are sufficient
    to answer the user's question.

    Args:
        question: Original user question.
        retrieved_docs: Documents retrieved from the Vector Database.

    Returns:
        EvaluationReport containing:
        - useful: whether the documents are sufficient
        - description: explanation of the decision
    """

    llm = LLM(
        model="gpt-4o-mini",
        temperature=0
    )

    prompt = f"""
You are an AI evaluator.

Your task is to determine whether the retrieved documents contain enough
information to answer the user's question.

User Question:
{question}

Retrieved Documents:
{retrieved_docs}

Return:
- useful = true if the documents are sufficient.
- useful = false if more information from the web is needed.

Explain your reasoning briefly.
"""

    result = llm.invoke(
        prompt,
        response_format=EvaluationReport
    )

    return result.content

In [8]:
docs = retrieve_game("Who published Minecraft?")

evaluation = evaluate_retrieval(
    "Who published Minecraft?",
    docs
)

print(evaluation)

{"useful":true,"description":"The retrieved documents contain clear information stating that Minecraft was published by Mojang Studios. This directly answers the user's question about who published Minecraft."}


#### Game Web Search Tool

In [9]:
@tool
def game_web_search(question: str):
    """
    Search the web for game-related information.

    Args:
        question: A question about the video game industry.

    Returns:
        Relevant web search results with citations.
    """

    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=3
    )

    return [
        {
            "title": r["title"],
            "content": r["content"],
            "url": r["url"]
        }
        for r in response["results"]
    ]

In [10]:
game_web_search(
    "Was Mortal Kombat X released for PlayStation 5?"
)

[{'title': 'Mortal Kombat X - IGN',
  'content': "Sep 29, 2021- It's almost the end of September, which means Sony has revealed the free games available next month for PlayStation Plus subscribers. On PS5, we're getting Hell Let Loose, and for PS4, there's PGA Tour 2K21 and Mortal Kombat X. Hell Let Loose is a brand new World War 2-themed FPS with large 100-player battles and an RTS-inspired meta game. It was released on PC this summer and is now making its way to consoles. PGA Tour 2K21 features 15 licensed PGA Tour courses and a robust player and course creator. Mortal Kombat X was originally released back in 2015, and we gave it a glowing review, calling it the best Mortal Kombat game, period. PS Plus subscribers can grab these games starting October 5th. Marvel's Spider-Man 2 may not be hitting PlayStation 5 consoles until 2023, but [...] 2 may not be hitting PlayStation 5 consoles until 2023, but Marvel is already teasing the game's darker storyline. On the This Week in Marvel pod

### Agent

In [11]:
instructions = """
You are UdaPlay, an AI Research Agent specializing in video games.

Workflow:
1. Always begin by calling retrieve_game to search the internal game database.
2. Always call evaluate_retrieval to determine whether the retrieved documents are sufficient.
3. If evaluate_retrieval indicates the documents are not useful, call game_web_search.
4. Prefer information from the internal database whenever possible.
5. Use web search only when necessary.
6. When web search is used, include the relevant source URLs in your answer.
7. Present answers clearly using bullet points whenever appropriate.
8. If previous conversation context is relevant, use it naturally.
9. Never fabricate information. If neither the database nor the web contains sufficient information, clearly state that.
"""
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search
    ],
    temperature=0
)

In [13]:
queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?"
]

for query in queries:
    print("=" * 80)
    print("QUESTION:", query)

    run = agent.invoke(query)

    final_state = run.get_final_state()

    print("\nANSWER:")
    print(final_state["messages"][-1].content)

QUESTION: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

ANSWER:
Pokémon Gold and Silver was released in **1999**. Here are some additional details:

- **Platform**: Game Boy Color
- **Genre**: Role-playing
- **Publisher**: Nintendo
- **Description**: These are second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
QUESTION: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

ANSWER:
The first 3D platformer Mario game is **Super Mario 64**. Here are some details:

- **Release Year**: 1996
- **Platform**: Nintendo 64
- **Genre**: Platformer
- **Publisher**: Nintendo
- **Description**: A groundbreaking 3D platformer that set ne

### (Optional) Advanced

In [14]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes